## Section 1: Environment Setup & Library Installation

In [ ]:
# Install necessary packages if running in Google Colab
!pip install -q scikit-learn pandas numpy matplotlib seaborn joblib ipywidgets

import os
import sys
import re
import json
import random
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Scikit-Learn modules
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold, cross_val_score
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

# Set fixed random seed for strict reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print('Setup complete. Random seed set to 42.')


## Section 2: Dataset Loading, Inspection & Duplicate Analysis


In [ ]:
DATASET_FILENAME = 'sarcasm_dataset.csv'

# Check if dataset exists or prompt upload in Colab
if not os.path.exists(DATASET_FILENAME):
    try:
        from google.colab import files
        print(f'{DATASET_FILENAME} not found. Please upload:')
        uploaded = files.upload()
    except ImportError:
        raise FileNotFoundError(f'Please ensure {DATASET_FILENAME} is in the working directory.')

df = pd.read_csv(DATASET_FILENAME)
print(f'Total records loaded: {len(df)}')
print(f'Columns present: {list(df.columns)}')

# Duplicate & Integrity Verification
exact_dups = df['sentence'].duplicated().sum()
null_counts = df.isnull().sum().to_dict()
unique_groups = df['group_id'].nunique()

print(f'Exact duplicate sentences: {exact_dups}')
print(f'Null values: {null_counts}')
print(f'Unique semantic groups (group_id): {unique_groups}')

# Verify group purity (each group must map to exactly one ground truth label)
group_label_counts = df.groupby('group_id')['label'].nunique()
impure_groups = (group_label_counts > 1).sum()
print(f'Impure multi-class groups: {impure_groups} (Must be 0)')
assert impure_groups == 0, 'Integrity Error: Found groups spanning multiple classes!'

df.head(5)


## Section 3: Exploratory Data Analysis (EDA) & Linguistic Profiling


In [ ]:
# Class distribution
print('Target Class Counts:')
class_counts = df['label'].value_counts()
print(class_counts)

# Sentence length statistics
df['char_length'] = df['sentence'].apply(len)
df['word_count'] = df['sentence'].apply(lambda s: len(str(s).split()))

print('\nSentence Length Statistics by Class:')
print(df.groupby('label')[['char_length', 'word_count']].describe())

# Visualizing EDA Distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Subplot 1: Class Balance
sns.barplot(x=class_counts.index, y=class_counts.values, ax=axes[0], palette=['#6366f1', '#10b981'])
axes[0].set_title('Class Balance (Target Variable)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Samples')
for i, v in enumerate(class_counts.values):
    axes[0].text(i, v + 15, str(v), ha='center', fontweight='bold')

# Subplot 2: Word Count Density
sns.kdeplot(data=df, x='word_count', hue='label', fill=True, common_norm=False, ax=axes[1], palette=['#6366f1', '#10b981'])
axes[1].set_title('Word Count Distribution by Class', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Word Count per Sentence')

# Subplot 3: Sub-Category Distribution
sub_counts = df['sub_category'].value_counts()
sns.barplot(y=sub_counts.index, x=sub_counts.values, ax=axes[2], palette='mako')
axes[2].set_title('Linguistic Sub-Category Breakdown', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Sample Count')

plt.tight_layout()
plt.show()


## Section 4: Text Normalization & Linguistic Signal Extraction


In [ ]:
POSITIVE_CUES = {
    'wah', 'mast', 'saras', 'perfect', 'superb', 'great', 'zabardast', 'jordar', 'kamaal',
    'shandar', 'relief', 'delicious', 'tasty', 'fresh', 'swadisht', 'fast', 'speed', 'bullet',
    'rocket', 'salute', 'award', 'nobel', 'congratulations', 'aabhar', 'thanks', 'thank',
    'satisfy', 'satisfied', 'happy', 'khush', 'khushi', 'maja', 'majja', 'maza', 'blessing',
    'genius', 'einstein', 'top', 'distinction', 'prashansaniya', 'clean', 'hygienic', 'smooth'
}

NEGATIVE_CUES = {
    'fail', 'waste', 'late', 'delay', 'thaki', 'kantali', 'kharaab', 'kharab', 'broken',
    'damage', 'damaged', 'shattered', 'tootyo', 'crash', 'stuck', 'fasaai', 'cancel',
    'cancelled', 'problem', 'takleef', 'dard', 'headache', 'sar', 'spoil', 'spoiled',
    'drop', 'dropped', 'burst', 'leak', 'cut', 'deduction', 'strike', 'disappointed',
    'sad', 'stress', 'tension', 'cheated', 'fraud', 'bad', 'loss', 'nuksan', 'potholes',
    'khadda', 'traffic', 'garmi', 'dirt', 'gandu', 'mativalu', 'ice', 'cold', 'thandu', 'burnt'
}

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        text = str(text or '')
    text = re.sub(r'\s+', ' ', text).strip()
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    return text

def extract_linguistic_signals(text: str) -> dict:
    cleaned = clean_text(text)
    tokens = set(re.findall(r'[a-zA-Z]+', cleaned.lower()))
    pos_found = sorted(list(tokens.intersection(POSITIVE_CUES)))
    neg_found = sorted(list(tokens.intersection(NEGATIVE_CUES)))
    sentiment_contrast = (len(pos_found) > 0 and len(neg_found) > 0)
    return {
        'cleaned_text': cleaned,
        'positive_cues': pos_found,
        'negative_cues': neg_found,
        'sentiment_contrast': sentiment_contrast,
        'exclamation_count': text.count('!'),
        'question_count': text.count('?')
    }

df['clean_sentence'] = df['sentence'].apply(clean_text)
print('Normalization completed. Sample output:')
print(df[['sentence', 'clean_sentence', 'label']].head(3))


## Section 5: Group-Aware Train / Val / Test Partitioning
**Eliminating Data Leakage**:
In code-mixed NLP, naive random splitting allows near-duplicate templates or spelling variants of the same sentence into both training and test sets. We use `GroupShuffleSplit` on `group_id` so that an entire semantic family resides strictly in one partition.

In [ ]:
X = df['clean_sentence']
y = df['label']
groups = df['group_id']

# Split 1: Hold out 15% of semantic groups for final Test Set
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=RANDOM_SEED)
train_val_idx, test_idx = next(gss_test.split(X, y, groups=groups))

# Split 2: Divide remaining 85% into Train (70% total) and Val (15% total -> 15/85 of train_val)
val_ratio = 0.15 / 0.85
gss_val = GroupShuffleSplit(n_splits=1, test_size=val_ratio, random_state=RANDOM_SEED)
train_val_groups = groups.iloc[train_val_idx]
train_sub_idx, val_sub_idx = next(
    gss_val.split(X.iloc[train_val_idx], y.iloc[train_val_idx], groups=train_val_groups)
)

train_idx = train_val_idx[train_sub_idx]
val_idx = train_val_idx[val_sub_idx]

train_groups = set(groups.iloc[train_idx])
val_groups = set(groups.iloc[val_idx])
test_groups = set(groups.iloc[test_idx])

# Strict Disjoint Assertions
assert len(train_groups & test_groups) == 0, 'Leakage detected between Train and Test!'
assert len(train_groups & val_groups) == 0, 'Leakage detected between Train and Val!'
assert len(val_groups & test_groups) == 0, 'Leakage detected between Val and Test!'

X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]

print('>> GROUP LEAKAGE AUDIT PASSED: ZERO GROUP OVERLAP.')
print(f'Train Set:      {len(X_train)} samples across {len(train_groups)} groups | Dist: {y_train.value_counts().to_dict()}')
print(f'Validation Set: {len(X_val)} samples across {len(val_groups)} groups | Dist: {y_val.value_counts().to_dict()}')
print(f'Held-Out Test:  {len(X_test)} samples across {len(test_groups)} groups | Dist: {y_test.value_counts().to_dict()}')


## Section 6: Baseline Model Implementation
We implement two simple baseline classifiers to benchmark against:
1. **Majority Class / Stratified Dummy Classifier**
2. **Simple Word-only Unigram Logistic Regression**

In [ ]:
# Baseline 1: Dummy Classifier
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)
dummy_val_pred = dummy.predict(X_val)
print(f'Baseline 1 (Majority Class Dummy) Val Accuracy: {accuracy_score(y_val, dummy_val_pred)*100:.2f}% | Macro F1: {f1_score(y_val, dummy_val_pred, average="macro")*100:.2f}%')

# Baseline 2: Simple Unigram Logistic Regression
simple_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 1), max_features=1000)),
    ('clf', LogisticRegression(random_state=RANDOM_SEED))
])
simple_pipe.fit(X_train, y_train)
simple_val_pred = simple_pipe.predict(X_val)
print(f'Baseline 2 (Unigram TF-IDF + Logistic Regression) Val Accuracy: {accuracy_score(y_val, simple_val_pred)*100:.2f}% | Macro F1: {f1_score(y_val, simple_val_pred, average="macro")*100:.2f}%')


## Section 7: Feature Engineering Pipeline (Word + Char-wb TF-IDF)
We build a dual-representation `FeatureUnion` combining:
- **Word-level TF-IDF (1 to 3 n-grams)**: Learns code-mixed phrases and sentiment idioms (*'wah re wah'*, *'what a relief'*).
- **Character-level word-boundary TF-IDF (3 to 5 n-grams)**: Captures subword phonetic stems invariant to transliteration differences (`chhe`/`che`, `shuu`/`su`).

In [ ]:
# Define dual-channel feature extractor
feature_union = FeatureUnion([
    ('word_tfidf', TfidfVectorizer(
        ngram_range=(1, 3),
        max_features=6000,
        sublinear_tf=True
    )),
    ('char_tfidf', TfidfVectorizer(
        analyzer='char_wb',
        ngram_range=(3, 5),
        max_features=12000,
        sublinear_tf=True
    ))
])
print('Feature engineering pipeline configured.')


## Section 8: Multi-Model Benchmark & Group-Stratified Cross-Validation
We benchmark 6 candidate machine learning models using `StratifiedGroupKFold(n_splits=5)` on the training set, and evaluate each on the validation set.

In [ ]:
candidates = {
    'Logistic Regression (C=2.0)': LogisticRegression(C=2.0, max_iter=1000, random_state=RANDOM_SEED),
    'Calibrated Linear SVM (C=1.0)': CalibratedClassifierCV(LinearSVC(C=1.0, random_state=RANDOM_SEED), cv=3),
    'Complement Naive Bayes (alpha=0.5)': ComplementNB(alpha=0.5),
    'Multinomial Naive Bayes (alpha=0.5)': MultinomialNB(alpha=0.5),
    'Random Forest (n_est=200)': RandomForestClassifier(n_estimators=200, random_state=RANDOM_SEED),
    'Soft Voting Ensemble': VotingClassifier(
        estimators=[
            ('lr', LogisticRegression(C=2.0, max_iter=1000, random_state=RANDOM_SEED)),
            ('svm', CalibratedClassifierCV(LinearSVC(C=1.0, random_state=RANDOM_SEED), cv=3)),
            ('cnb', ComplementNB(alpha=0.5))
        ],
        voting='soft'
    )
}

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
train_group_series = groups.iloc[train_idx].reset_index(drop=True)
X_train_reset = X_train.reset_index(drop=True)
y_train_reset = y_train.reset_index(drop=True)

benchmark_results = []
trained_pipelines = {}

print('Running Group-Stratified 5-Fold Cross-Validation on Training Set...')
for name, clf in candidates.items():
    pipe = Pipeline([
        ('features', feature_union),
        ('classifier', clf)
    ])
    cv_scores = cross_val_score(
        pipe, X_train_reset, y_train_reset,
        cv=sgkf.split(X_train_reset, y_train_reset, groups=train_group_series),
        scoring='f1_macro'
    )
    cv_mean = float(np.mean(cv_scores))
    cv_std = float(np.std(cv_scores))

    # Fit on training split
    pipe.fit(X_train, y_train)
    trained_pipelines[name] = pipe

    # Evaluate on validation split
    y_val_pred = pipe.predict(X_val)
    val_acc = float(accuracy_score(y_val, y_val_pred))
    val_f1 = float(f1_score(y_val, y_val_pred, average='macro'))

    benchmark_results.append({
        'Model': name,
        'CV_Macro_F1': cv_mean,
        'CV_Std': cv_std,
        'Val_Accuracy': val_acc,
        'Val_Macro_F1': val_f1
    })
    print(f' - {name:<35} | CV Macro-F1: {cv_mean:.4f} (+/- {cv_std:.4f}) | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}')

results_df = pd.DataFrame(benchmark_results).sort_values(by='Val_Macro_F1', ascending=False).reset_index(drop=True)
results_df


## Section 9: Model Selection & Hyperparameter Tuning
We identify the top candidate architecture based on validation macro F1-score and inspect tuning decisions.

In [ ]:
# Visualize benchmark results
plt.figure(figsize=(12, 5))
x_indices = np.arange(len(results_df))
bar_width = 0.35

plt.bar(x_indices - bar_width/2, results_df['CV_Macro_F1'], bar_width, label='5-Fold CV Macro-F1', color='#6366f1')
plt.bar(x_indices + bar_width/2, results_df['Val_Macro_F1'], bar_width, label='Validation Macro-F1', color='#10b981')

plt.xticks(x_indices, results_df['Model'], rotation=25, ha='right', fontsize=10)
plt.ylabel('Macro F1-Score', fontsize=12)
plt.title('Benchmark Architecture Comparison (Group-Aware CV vs Validation)', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

winner_name = results_df.iloc[0]['Model']
print(f'Selected Best Architecture: {winner_name}')


## Section 10: Final Evaluation on the Untouched Held-Out Test Set
We evaluate the selected model **strictly once** on the untouched held-out test partition (unseen semantic groups).

In [ ]:
best_pipe = trained_pipelines[winner_name]
y_test_pred = best_pipe.predict(X_test)

test_acc = accuracy_score(y_test, y_test_pred)
test_prec = precision_score(y_test, y_test_pred, average='macro')
test_rec = recall_score(y_test, y_test_pred, average='macro')
test_f1 = f1_score(y_test, y_test_pred, average='macro')

print('=' * 65)
print(f'HELD-OUT TEST SET EVALUATION ({winner_name})')
print('=' * 65)
print(f'Accuracy:        {test_acc * 100:.2f}%')
print(f'Macro Precision: {test_prec * 100:.2f}%')
print(f'Macro Recall:    {test_rec * 100:.2f}%')
print(f'Macro F1-Score:  {test_f1 * 100:.2f}%')

print('\nDetailed Classification Report:')
print(classification_report(y_test, y_test_pred, digits=4))

# Confusion Matrix Plot
cm = confusion_matrix(y_test, y_test_pred, labels=['Non-Sarcasm', 'Sarcasm'])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Non-Sarcasm', 'Sarcasm'], yticklabels=['Non-Sarcasm', 'Sarcasm'])
plt.title(f'Confusion Matrix on Held-Out Test Data\n(Accuracy: {test_acc*100:.2f}%)', fontsize=12, fontweight='bold')
plt.ylabel('Actual Ground Truth')
plt.xlabel('Model Prediction')
plt.tight_layout()
plt.show()


## Section 11: In-Depth Error Analysis & Provenance Breakdown
We inspect False Positives (genuine statements misclassified as sarcastic) and False Negatives (sarcastic statements missed by the model), and break down accuracy across linguistic subcategories and source types.

In [ ]:
test_eval_df = df.iloc[test_idx].copy().reset_index(drop=True)
test_eval_df['predicted'] = y_test_pred
test_eval_df['correct'] = (test_eval_df['label'] == test_eval_df['predicted'])

if hasattr(best_pipe, 'predict_proba'):
    sarc_idx = list(best_pipe.classes_).index('Sarcasm')
    test_eval_df['sarcasm_prob'] = best_pipe.predict_proba(X_test)[:, sarc_idx]
else:
    test_eval_df['sarcasm_prob'] = np.nan

# Performance by Linguistic Sub-Category
print('Accuracy by Sub-Category:')
for sub, grp in test_eval_df.groupby('sub_category'):
    print(f' - {sub:<25} ({len(grp):>3} samples): {grp["correct"].mean() * 100:.1f}%')

# Performance by Source Provenance
print('\nAccuracy by Source Type:')
for src, grp in test_eval_df.groupby('source_type'):
    print(f' - {src:<25} ({len(grp):>3} samples): {grp["correct"].mean() * 100:.1f}%')

# False Positives
fps = test_eval_df[(test_eval_df['label'] == 'Non-Sarcasm') & (test_eval_df['predicted'] == 'Sarcasm')]
print(f'\nFalse Positives Count: {len(fps)}')
for _, r in fps.head(5).iterrows():
    print(f' - "{r["sentence"]}" (Sub: {r["sub_category"]}, P(Sarc): {r["sarcasm_prob"]:.2f})')

# False Negatives
fns = test_eval_df[(test_eval_df['label'] == 'Sarcasm') & (test_eval_df['predicted'] == 'Non-Sarcasm')]
print(f'\nFalse Negatives Count: {len(fns)}')
for _, r in fns.head(5).iterrows():
    print(f' - "{r["sentence"]}" (Sub: {r["sub_category"]}, P(Sarc): {r["sarcasm_prob"]:.2f})')


## Section 12: Model Serialization, Artifact Packaging & Live Demo
We serialize the pipeline into `sarcasm_pipeline.joblib`, generate `model_metadata.json`, and run interactive predictions on the core benchmark examples.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Callable prediction helper using the trained pipeline and training preprocessor.
def predict_sentence(text):
    cleaned = clean_text(text)
    if not cleaned.strip():
        raise ValueError("Enter a non-empty sentence.")
    prediction = best_pipe.predict([cleaned])[0]
    result = {"prediction": prediction}
    if hasattr(best_pipe, "predict_proba"):
        sarcasm_index = list(best_pipe.classes_).index("Sarcasm")
        result["sarcasm_probability"] = float(best_pipe.predict_proba([cleaned])[0][sarcasm_index])
    result["linguistic_signals"] = extract_linguistic_signals(text)
    return result

interactive_input = widgets.Textarea(
    value="", placeholder="Type a Gujarati-English Romanized sentence…",
    description="Sentence:", layout=widgets.Layout(width="100%", height="90px")
)
interactive_button = widgets.Button(description="Detect Sarcasm", button_style="primary")
interactive_output = widgets.Output()

def _on_predict_clicked(_):
    with interactive_output:
        clear_output()
        try:
            result = predict_sentence(interactive_input.value)
            print("Predicted class:", result["prediction"])
            if "sarcasm_probability" in result:
                print("Sarcasm probability:", f"{result['sarcasm_probability']:.1%}")
            print("Linguistic diagnostics:", result["linguistic_signals"])
        except ValueError as exc:
            print(exc)

interactive_button.on_click(_on_predict_clicked)
display(interactive_input, interactive_button, interactive_output)